# Finance AI Poster Outputs

This notebook reads the processed finance results and creates poster-ready tables and figures.

It also copies the main assets into a top-level `poster_assets` folder so they are easy to find in Finder.


In [ ]:
%pip install pandas matplotlib seaborn openpyxl


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

ROOT = Path('/Users/stephenye/stephen/coding/project/datajam')
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports'
FIGURES_DIR = ROOT / 'figures'
POSTER_ASSETS_DIR = ROOT / 'poster_assets'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
POSTER_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

focus_path = PROCESSED_DIR / 'finance_focus_jobs.csv'
focus = pd.read_csv(focus_path)

# Fill the missing Credit Analysts BLS-linked values directly in case the
# analysis notebook has not been rerun yet.
credit_mask = focus['soc_code'] == '13-2041'
focus.loc[credit_mask, 'median_annual_wage_2024'] = focus.loc[credit_mask, 'median_annual_wage_2024'].fillna(80970)
focus.loc[credit_mask, 'projected_growth_pct_2024_2034'] = focus.loc[credit_mask, 'projected_growth_pct_2024_2034'].fillna(-4)
focus.loc[credit_mask, 'source_url'] = focus.loc[credit_mask, 'source_url'].fillna('https://www.onetonline.org/link/summary/13-2041.00')

# Keep only ranked rows for poster visuals.
poster_df = focus.dropna(subset=['ai_enhancement_score']).copy()
poster_df = poster_df.sort_values('ai_enhancement_score', ascending=False).reset_index(drop=True)
poster_df['focus_rank'] = range(1, len(poster_df) + 1)

palette = {'analytical': '#c44e7e', 'routine': '#7aa6c2'}


In [ ]:
# Summary table
summary_table = poster_df[[
    'focus_rank',
    'occupation_title',
    'group_label',
    'ai_exposure',
    'median_annual_wage_2024',
    'projected_growth_pct_2024_2034',
    'ai_enhancement_score',
]].copy()

summary_table = summary_table.rename(columns={
    'focus_rank': 'Rank',
    'occupation_title': 'Occupation',
    'group_label': 'Group',
    'ai_exposure': 'AI Exposure',
    'median_annual_wage_2024': 'Median Wage (2024)',
    'projected_growth_pct_2024_2034': 'Growth % (2024-34)',
    'ai_enhancement_score': 'Enhancement Score',
})

summary_table = summary_table.round({
    'AI Exposure': 3,
    'Median Wage (2024)': 0,
    'Growth % (2024-34)': 1,
    'Enhancement Score': 3,
})

summary_table.to_csv(REPORTS_DIR / 'finance_summary_table.csv', index=False)
summary_table


In [ ]:
# Figure 1: ranked enhancement bar chart
plot_df = poster_df.sort_values('ai_enhancement_score', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(
    plot_df['occupation_title'],
    plot_df['ai_enhancement_score'],
    color=[palette[g] for g in plot_df['group_label']],
)
plt.xlabel('AI Enhancement Score')
plt.ylabel('Occupation')
plt.title('Estimated AI Enhancement Across Finance Occupations')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'finance_ai_enhancement_bar.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 2: AI exposure vs projected growth
scatter_df = poster_df.dropna(subset=['ai_exposure', 'projected_growth_pct_2024_2034']).copy()
scatter_df['bubble_size'] = scatter_df['median_annual_wage_2024'].fillna(scatter_df['median_annual_wage_2024'].median()) / 220

plt.figure(figsize=(9, 6))
plt.scatter(
    scatter_df['ai_exposure'],
    scatter_df['projected_growth_pct_2024_2034'],
    s=scatter_df['bubble_size'],
    c=scatter_df['group_label'].map(palette),
    edgecolors='black',
    linewidths=0.8,
    alpha=0.85,
)

for _, row in scatter_df.iterrows():
    plt.text(
        row['ai_exposure'] + 0.003,
        row['projected_growth_pct_2024_2034'] + 0.15,
        row['occupation_title'],
        fontsize=8,
    )

x_pad = 0.03
y_pad = 1.0
plt.xlim(scatter_df['ai_exposure'].min() - x_pad, scatter_df['ai_exposure'].max() + x_pad)
plt.ylim(scatter_df['projected_growth_pct_2024_2034'].min() - y_pad, scatter_df['projected_growth_pct_2024_2034'].max() + y_pad)
plt.xlabel('AI Exposure (AIOE)')
plt.ylabel('Projected Growth % (2024-34)')
plt.title('AI Exposure vs Projected Growth for Finance Occupations')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'finance_ai_scatter.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 3: standardized feature heatmap
heatmap_df = poster_df[[
    'occupation_title',
    'ai_exposure',
    'cognitive_skill_level',
    'finance_knowledge_level',
    'routine_activity_level',
    'median_annual_wage_2024',
    'projected_growth_pct_2024_2034',
    'ai_enhancement_score',
]].copy()

heatmap_df = heatmap_df.set_index('occupation_title')
heatmap_df = heatmap_df.dropna(axis=1, how='all')
heatmap_df = heatmap_df.apply(lambda col: (col - col.mean()) / col.std(ddof=0), axis=0)
heatmap_df = heatmap_df.replace([float('inf'), -float('inf')], pd.NA).dropna(axis=1, how='all')

plt.figure(figsize=(11, 6))
sns.heatmap(heatmap_df, cmap='RdPu', center=0, linewidths=0.5)
plt.title('Standardized Feature Heatmap Across Finance Occupations')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'finance_feature_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 4: correlation heatmap
corr_df = poster_df[[
    'ai_exposure',
    'cognitive_skill_level',
    'finance_knowledge_level',
    'routine_activity_level',
    'median_annual_wage_2024',
    'projected_growth_pct_2024_2034',
    'ai_enhancement_score',
]].copy()

corr_df = corr_df.dropna(axis=1, how='all')
corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Finance AI Features')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'finance_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Group comparison table
group_comparison = poster_df.groupby('group_label')[[
    'ai_exposure',
    'median_annual_wage_2024',
    'projected_growth_pct_2024_2034',
    'ai_enhancement_score',
]].mean(numeric_only=True).round(3)

group_comparison.to_csv(REPORTS_DIR / 'finance_group_comparison.csv')
group_comparison


In [ ]:
# Make a top-level asset folder that is easy to find in Finder
focus.to_csv(POSTER_ASSETS_DIR / 'finance_focus_jobs.csv', index=False)
summary_table.to_csv(POSTER_ASSETS_DIR / 'finance_summary_table.csv', index=False)
group_comparison.to_csv(POSTER_ASSETS_DIR / 'finance_group_comparison.csv')

for file_name in [
    'finance_ai_enhancement_bar.png',
    'finance_ai_scatter.png',
    'finance_feature_heatmap.png',
    'finance_correlation_heatmap.png',
]:
    shutil.copy2(FIGURES_DIR / file_name, POSTER_ASSETS_DIR / file_name)

for file_name in [
    'finance_job_ranking.md',
    'finance_summary_table.csv',
    'finance_group_comparison.csv',
]:
    source = REPORTS_DIR / file_name
    if source.exists():
        shutil.copy2(source, POSTER_ASSETS_DIR / file_name)

sorted(p.name for p in POSTER_ASSETS_DIR.iterdir())
